# Direct solvers and static condensation

First we measure how the stiffness matrix changes under refinement. Then we solve one Poisson problem twice, factoring the full free matrix and factoring a smaller matrix on the element interfaces.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from ngsolve import *
from ngsolve.meshes import Make1DMesh, MakeStructured2DMesh

## Conditioning on a uniform mesh

For piecewise linears on $(0,1)$ with zero endpoint values, a mesh of $n$ intervals has $N=n-1$ free coefficients and mesh size $h=1/n$. On the free subspace the eigenvalues of the stiffness matrix are

$$
\begin{align*}
\lambda_j(A) = \frac{4}{h}\sin^2\!\left(\frac{j\pi h}{2}\right),
\qquad j=1,\ldots,N.
\end{align*}
$$

This predicts $\kappa_2(A)=\lambda_N/\lambda_1\sim 4/(\pi^2 h^2)$. For these small matrices we can compute the eigenvalues directly. What happens to the condition number when we halve $h$?

In [ ]:
print("  n    kappa(A)    h^2 kappa(A)")
for n in [8, 16, 32, 64, 128]:
    line = Make1DMesh(n)
    V = H1(line, order=1, dirichlet=".*")
    u, v = V.TnT()
    a = BilinearForm(grad(u) * grad(v) * dx, symmetric=True).Assemble()

    free = list(V.FreeDofs())
    A = np.array(a.mat.ToDense())
    A_line = A[free][:, free]
    eigenvalues = np.linalg.eigvalsh(A_line)
    kappa = eigenvalues[-1] / eigenvalues[0]
    print(f"{n:3d}  {kappa:10.2f}  {kappa / n**2:14.4f}")

## Why does the ordering matter?

Cholesky writes the free stiffness matrix as $A=LL^\top$. On the line, eliminating the leftmost remaining vertex leaves a shorter chain. Eliminating an interior vertex first couples its two neighbors, creating a new nonzero in the factor (called **fill-in**). We can see this directly on the last mesh above. Compare the natural order with one that eliminates alternating interior vertices. We compute the small dense factors to inspect their sparsity. Gray marks the original lower triangle and red marks entries created during elimination, the fill-in.

In [ ]:
N = len(A_line)
natural = np.arange(N)
alternating = np.r_[natural[1::2], natural[::2]]

print(f"Natural order: {natural}")
print(f"Alternating order: {alternating}")

fig, axes = plt.subplots(1, 2, figsize=(7, 3.4), layout="constrained")
for ax, order, name in zip(axes, (natural, alternating), ("Left to right", "Alternating")):
    ordered = A_line[np.ix_(order, order)]
    L = np.linalg.cholesky(ordered)
    original = np.tril(ordered) != 0
    fill = (np.abs(L) > 1e-12) & ~original
    ax.spy(original, color="0.3", markersize=2)
    ax.spy(fill, color="tab:red", markersize=3)
    ax.set_title(f"{name}\n{np.count_nonzero(fill)} fill entries")
plt.show()

## A sparse direct solve on the square

On $\Omega=(0,1)^2$ take $u(x,y)=x(1-x)y(1-y)$, which vanishes on the boundary. With $f=-\Delta u$, the finite element problem is to find $u_h\in V_h\subset H^1_0(\Omega)$ such that

$$
\begin{align*}
\int_\Omega \nabla u_h\cdot\nabla v_h\,dx
  = \int_\Omega f v_h\,dx \qquad \text{for all }v_h\in V_h.
\end{align*}
$$

We use $Q_2$ elements on nine squares. The exact solution is quadratic in each coordinate, so $u$ belongs to this space.

In [ ]:
mesh = MakeStructured2DMesh(nx=3, ny=3, quads=True)
fes = H1(mesh, order=2, dirichlet=".*")
u, v = fes.TnT()
stiffness = grad(u) * grad(v) * dx

u_exact = x * (1 - x) * y * (1 - y)
f = 2 * (x * (1 - x) + y * (1 - y))
a = BilinearForm(stiffness, symmetric=True).Assemble()
load = LinearForm(f * v * dx).Assemble()

The matrix contains rows for prescribed boundary coefficients. `FreeDofs()` selects the rows and columns to factor. Applying the sparse Cholesky factorization solves for the free coefficients and leaves zero values on the boundary.

The factorization is built once. For each right-hand side the solver applies the two triangular solves $Ly=b$ and $L^\top U=y$ in the factor ordering. We reuse the same factor for a second load and check its residual on the free space.

In [ ]:
free = fes.FreeDofs()
inverse = a.mat.Inverse(free, inverse="sparsecholesky")
uh = GridFunction(fes)
uh.vec.data = inverse * load.vec

error = Integrate((uh - u_exact)**2, mesh, order=8)**0.5
print(f"Free coefficients: {sum(free)}")
print(f"L2 error: {error:.2e}")

second_load = LinearForm(v * dx).Assemble()
U_second = (inverse * second_load.vec).Evaluate()
P = Projector(free, True)
residual = P * (second_load.vec - a.mat * U_second)
print(f"Relative residual for f = 1: {Norm(residual) / Norm(P * second_load.vec):.2e}")

## Eliminate the cell interiors

Each square has one $Q_2$ basis function supported on that square and vanishing on its boundary. Write $L$ for the interior coefficients and $E$ for the free interface coefficients. The interior equations give $U_L=A_{LL}^{-1}(b_L-A_{LE}U_E)$. Substitution into the interface equations gives

$$
\begin{align*}
S U_E &= \widetilde b_E,\\
S &= A_{EE}-A_{EL}A_{LL}^{-1}A_{LE},\\
\widetilde b_E &= b_E-A_{EL}A_{LL}^{-1}b_L.
\end{align*}
$$

`condense=True` eliminates the interior coefficients element by element during assembly. `harmonic_extension_trans` supplies the correction to the load. We solve on the free interface coefficients selected by `FreeDofs(coupling=True)`.

In [ ]:
condensed = BilinearForm(stiffness, symmetric=True, condense=True).Assemble()
rhs = load.vec.CreateVector()
rhs.data = load.vec + condensed.harmonic_extension_trans * load.vec

interface = fes.FreeDofs(coupling=True)
schur_inverse = condensed.mat.Inverse(interface, inverse="sparsecholesky")
interface_values = load.vec.CreateVector()
interface_values.data = schur_inverse * rhs

print(f"Free interface coefficients: {sum(interface)}")
print(f"Eliminated interior coefficients: {sum(free) - sum(interface)}")

### How does condensation change the sparsity pattern?

On the left, we order the free coefficients as $(E,L)$. The lines separate the interface block $A_{EE}$ in the upper left from the rows and columns of the cell interiors. On the right, we plot the Schur complement $S$ in the same interface ordering.

Compare the upper-left block with the right panel. Each red square marks a coupling created by eliminating a cell interior: two interface coefficients can interact through that interior even when their entry in $A_{EE}$ is zero. Only coefficients belonging to a common cell can acquire such a coupling.

In [ ]:
on_interface = np.array(list(interface))
in_cell = np.array(list(free)) & ~on_interface
interface_ids = np.flatnonzero(on_interface)
order = np.r_[interface_ids, np.flatnonzero(in_cell)]
n_interface = len(interface_ids)

A_ordered = np.array(a.mat.ToDense())[np.ix_(order, order)]
S = np.array(condensed.mat.ToDense())[np.ix_(interface_ids, interface_ids)]
threshold = 1e-12 * np.max(np.abs(A_ordered))  # Ignore roundoff on this small mesh.
full_pattern = np.abs(A_ordered) > threshold
schur_pattern = np.abs(S) > threshold
interface_pattern = full_pattern[:n_interface, :n_interface]
new_entries = schur_pattern & ~interface_pattern

fig, (left, right) = plt.subplots(1, 2, figsize=(8, 4), layout="constrained")
left.spy(full_pattern, color="0.25", markersize=4)
left.axhline(n_interface - 0.5, color="0.65", linewidth=1)
left.axvline(n_interface - 0.5, color="0.65", linewidth=1)
left.set_title(f"Full system ($E$, then $L$)\n{len(order)} unknowns")
right.spy(schur_pattern & interface_pattern, color="0.25", markersize=5)
right.spy(new_entries, color="tab:red", markersize=5)
right.set_title(f"Schur complement ($E$)\n{n_interface} unknowns, {new_entries.sum()} new entries")
plt.show()

## Recover the interior coefficients

The local recovery formula has two contributions. `harmonic_extension` applies $-A_{LL}^{-1}A_{LE}$ to the interface values. `inner_solve` applies $A_{LL}^{-1}$ to the original cell loads. After adding both, we can compare every coefficient with the full solve.

In [ ]:
uh_condensed = GridFunction(fes)
uh_condensed.vec.data = interface_values + condensed.harmonic_extension * interface_values
uh_condensed.vec.data += condensed.inner_solve * load.vec

difference = Norm(uh_condensed.vec - uh.vec)
print(f"Euclidean norm of the coefficient difference: {difference:.2e}")